In [ ]:
!pip install lightgbm scikit-learn pandas numpy joblib

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import VotingClassifier
from lightgbm import LGBMClassifier
import joblib

class VotingLGBMEnsemble(VotingClassifier):
    @property
    def feature_importances_(self):
        return np.mean([est.feature_importances_ for est in self.estimators_], axis=0)

In [ ]:
print("Downloading real Kaggle Credit Card Fraud dataset...")
url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df_raw = pd.read_csv(url)

print("Mapping complex latent features to app-specific feature ranges...")
df = pd.DataFrame()

# V1 spans roughly -56 to +5. Map to transaction_count (mostly 50-150)
df["transaction_count"] = np.clip(100 + df_raw["V1"] * 15, 1, 5000).astype(int)

# Amount -> avg_transaction_amount (scaled up for Rupee realism)
df["avg_transaction_amount"] = df_raw["Amount"] * 80.0 + 10.0

# V2 -> KYC Tier (0, 1, 2, 3)
df["kyc_tier"] = np.clip(np.digitize(df_raw["V2"], bins=[-2, 0, 2]), 0, 3)

# V3 -> device_trust_score (Sigmoid to bind between 0.0 and 1.0)
def sigmoid(x): return 1 / (1 + np.exp(-x))
df["device_trust_score"] = sigmoid(df_raw["V3"])

# Time -> days_since_registration (spread seconds out over months)
df["days_since_registration"] = np.clip((df_raw["Time"] / 3600) * 5, 1, 2000).astype(int)

# V4 -> fraud_flags (Higher V4 correlates heavily with fraud in the real dataset)
df["fraud_flags"] = np.clip(np.digitize(df_raw["V4"], bins=[1, 3, 5]), 0, 3)

# total_spent -> Derived naturally
df["total_spent"] = df["transaction_count"] * df["avg_transaction_amount"]

# Target: Class is natively 1 for fraud, 0 for normal in this dataset
df["is_risky"] = df_raw["Class"]

print(f"Dataset mathematically mapped. Total samples: {len(df)}")
print(f"Fraud cases: {df['is_risky'].sum()} out of {len(df)}")

In [ ]:
feature_cols = [
    "transaction_count", "avg_transaction_amount", "kyc_tier",
    "device_trust_score", "days_since_registration", "fraud_flags",
    "total_spent",
]

X = df[feature_cols].values
y = df["is_risky"].values

# Note: The Kaggle dataset is highly imbalanced (0.17% fraud).
# LGBM can handle this, but stratifying is crucial.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
lgbm1 = LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, is_unbalance=True, random_state=42, verbose=-1)
lgbm2 = LGBMClassifier(n_estimators=200, learning_rate=0.01, max_depth=6, scale_pos_weight=10, random_state=42, verbose=-1)
lgbm3 = LGBMClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, is_unbalance=True, random_state=42, verbose=-1)

model = VotingLGBMEnsemble(
    estimators=[
        ('lgbm1', lgbm1),
        ('lgbm2', lgbm2),
        ('lgbm3', lgbm3)
    ],
    voting='soft'
)

print("Training Ensemble of LGBM Classifiers...")
model.fit(X_train, y_train)
print("Training complete.")

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n=== Model Evaluation ===")
print(classification_report(y_test, y_pred, target_names=["Safe", "Risky"]))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

print("\n=== Feature Importances ===")
for name, imp in sorted(
    zip(feature_cols, model.feature_importances_),
    key=lambda x: -x[1],
):
    print(f"  {name:30s}: {imp:.4f}")

In [ ]:
joblib.dump(model, "risk_model.joblib")
print("Model saved successfully to 'risk_model.joblib'")